In [1]:
import os
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [9]:
# Error keywords for flagging
ERROR_KEYWORDS = ["SQL syntax", "warning", "exception", "error", "mysql", "pgsql", "sqlite"]
# Load existing dataset
existing_df = pd.read_csv('metadata.csv')  # Replace with your file path

In [10]:
# Prepare output directory
os.makedirs('data', exist_ok=True)
os.makedirs('outputs/response_logs', exist_ok=True)

In [11]:
# List to hold new rows
new_rows = []

In [12]:
# Session for persistent cookies (e.g., for DVWA login)
session = requests.Session()

In [13]:
# Function to check for error messages
def has_error_message(response_text):
    return 1 if any(keyword.lower() in response_text.lower() for keyword in ERROR_KEYWORDS) else 0

In [14]:
# Function to send request and collect data
def send_request(url, method, param_name, input_value, form_action, label):
    start_time = time.time()

    if method.upper() == 'GET':
        params = {param_name: input_value}
        response = session.get(url, params=params)
    elif method.upper() == 'POST':
        # Construct the full URL for the POST request
        if not form_action.startswith(('http://', 'https://')):
            # Assuming form_action is relative to the base url
            parsed_url = requests.utils.urlparse(url)
            form_url = f"{parsed_url.scheme}://{parsed_url.netloc}/{form_action}"
        else:
            form_url = form_action

        data = {param_name: input_value}
        response = session.post(form_url, data=data)
    else:
        raise ValueError(f"Unsupported method: {method}")

    response_time = (time.time() - start_time) * 1000  # in ms
    html_content_length = len(response.text)
    error_message_flag = has_error_message(response.text)

    # Optional: Save full response body
    response_path = f"outputs/response_logs/response_{len(new_rows)}.html"
    with open(response_path, 'w', encoding='utf-8') as f:
        f.write(response.text)

    return {
        'url': url,
        'method': method,
        'param_name': param_name,
        'input_type': row['input_type'],  # From existing
        'default_value': row['default_value'],
        'form_action': form_action,
        'input_value': input_value,
        'label': label,
        'response_status': response.status_code,
        'response_time': response_time,
        'html_content_length': html_content_length,
        'error_message_flag': error_message_flag,
        'response_body_path': response_path  # Optional
    }

# Iterate over each row in existing dataset
for _, row in existing_df.iterrows():
    url = row['url']
    method = row['method']
    param_name = row['param_name']
    form_action = row['form_action']

    # Send normal requests (use default_value or simple benign)
    normal_value = row['default_value'] if pd.notna(row['default_value']) else 'test'
    new_rows.append(send_request(url, method, param_name, normal_value, form_action, 'safe'))

    # Send injection requests (loop over a few payloads)
    for inj_payload in ["' OR '1'='1' --", "1; DROP TABLE users --"]:  # Add more as needed
        new_rows.append(send_request(url, method, param_name, inj_payload, form_action, 'injection'))

# Create enhanced DataFrame and save
enhanced_df = pd.DataFrame(new_rows)
enhanced_df.to_csv('data/raw_server_responses.csv', index=False)

print("Task complete! Enhanced dataset saved to data/raw_server_responses.csv")

Task complete! Enhanced dataset saved to data/raw_server_responses.csv
